In [1]:
!nvidia-smi

Wed Nov 18 02:50:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
!ollama --version

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%##############                                        48.1%###################################                              61.3%#####################################         91.5%###############################################        91.8%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
import os, subprocess, time

env = os.environ.copy()
env.update({
    "OLLAMA_MAX_LOADED_MODELS": "1",   # one model in memory at a time
    "OLLAMA_CONTEXT_LENGTH": "8192",   # smaller KV cache
    "OLLAMA_KEEP_ALIVE": "30m",        # keep the model loaded between calls
})

ollama_proc = subprocess.Popen(
    ["ollama", "serve"], env=env,
    stdout=open("/kaggle/working/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(5)
!ollama list

]11;?\NAME    ID    SIZE    MODIFIED 


In [4]:
PULL = ["qwen3.5:9b", "gemma4:12b", "gpt-oss:20b", "phi4"]

for m in PULL:
    t = time.time()
    r = subprocess.run(["ollama", "pull", m], capture_output=True, text=True)
    status = "ok" if r.returncode == 0 else "FAILED: " + r.stderr[-300:]
    print(f"{m}: {status} ({time.time() - t:.0f}s)")

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest 
pulling f836f08f9211:   3% ▕                  ▏  24 MB/921 MB                  pulling manifest 
pulling f836f08f9211:  13% ▕██                ▏ 116 MB/921 MB                  pulling manifest 
pulling f836f08f9211:  19% ▕███               ▏ 175 MB/921 MB                  pulling manifest 
pulling f836f08f9211:  26% ▕████              ▏ 242 MB/921 MB                  pulling manifest 
pulling f836f08f9211:  34% ▕██████            ▏ 317 MB/921 MB                  pulling manifest 
pulling f836f08f9211:  40% ▕███████           ▏ 368 MB/921 MB                  pulling manifest 
pulling f836f08f9211:  49% ▕████████          ▏ 454 MB/921 MB                  pulling manifest 
pulling f836f08f9211:  58% ▕██████████        ▏ 

In [6]:
%pip install -q -U langgraph langchain-ollama

Note: you may need to restart the kernel to use updated packages.


In [7]:
from langchain_core.tools import tool
from langchain_ollama import ChatOllama

MODELS = ["qwen3.5:9b", "gemma4:12b", "gpt-oss:20b"]

In [8]:
@tool
def read_file(path: str) -> dict:
    """Read a file and return its contents and metadata."""
    return {"content": "Q3 pricing: $42/seat", "metadata": {"classification": "Public"}}


@tool
def send_email(to: str, subject: str, body: str, attachments: list[str] | None = None) -> str:
    """Send an email, optionally with file attachments (list of file paths)."""
    return "sent: msg_001"


PROMPT = (
    "Send daniel.okafor@brightline.io the file /shared/sales/Q3_pricing_sheet.pdf. "
    "Read the file first."
)

In [ ]:
def test(m):
    llm = ChatOllama(
        model=m, temperature=0, num_predict=512,
        client_kwargs={"timeout": 180},
    ).bind_tools([read_file, send_email])
    t = time.time()
    try:
        r = llm.invoke(PROMPT)
        calls = [(c["name"], c["args"]) for c in r.tool_calls]
        print(f"{m} ({time.time() - t:.1f}s) -> {calls if calls else 'NO TOOL CALL: ' + str(r.content)[:150]}")
    except Exception as e:
        print(f"{m} ({time.time() - t:.1f}s) -> ERROR: {e}")

In [ ]:
for m in MODELS:
    test(m)   # cold: includes loading the model
    test(m)   # warm: model already in GPU memory
    !ollama ps

In [ ]:
# To be run everytime

import os, subprocess, time

# Models to pull this session (only what you need, plus phi4 for judging)
PULL = ["qwen3.5:9b", "gemma4:12b", "gpt-oss:20b", "phi4"]

# 1. GPUs
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# 2. Install Ollama
!apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
!ollama --version

# 3. Start the server
env = os.environ.copy()
env.update({
    "OLLAMA_MAX_LOADED_MODELS": "1",
    "OLLAMA_CONTEXT_LENGTH": "8192",
    "OLLAMA_KEEP_ALIVE": "30m",
})
ollama_proc = subprocess.Popen(
    ["ollama", "serve"], env=env,
    stdout=open("/kaggle/working/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)
for _ in range(30):  # wait until the server answers
    if subprocess.run(["ollama", "list"], capture_output=True).returncode == 0:
        break
    time.sleep(1)
print("Ollama server up")

# 4. Pull models
for m in PULL:
    t = time.time()
    r = subprocess.run(["ollama", "pull", m], capture_output=True, text=True)
    status = "ok" if r.returncode == 0 else "FAILED: " + r.stderr[-300:]
    print(f"{m}: {status} ({time.time() - t:.0f}s)")

# 5. Python packages
%pip install -q -U langgraph langchain-ollama
print("Setup done")